# **import og df udvvælgelse**

In [31]:
import pandas as pd
import gender_guesser.detector as gender
import spacy
from sentida import Sentida
import geonamescache
import re
import json
import requests
from pathlib import Path


## indlæs og udvælg fra df

In [32]:

el = pd.read_parquet('elgiganten.parquet', engine='pyarrow')
power = pd.read_parquet('power.parquet', engine='pyarrow')

el['store'] = 'elgiganten'
power['store'] = 'power'
df = pd.concat([el, power], ignore_index=True)

elgiganten = df[df["store"] == "elgiganten"].iloc[:2500]
power      = df[df["store"] == "power"].iloc[:2500]

df = pd.concat([elgiganten, power]).reset_index(drop=True)


# **feature engineering**

## rating

In [33]:
df["rating_1_til_5"]=df["rating"]//10 

## gender

In [34]:
d = gender.Detector(case_sensitive=False)


df["gender"]=df["name"].apply(lambda x: d.get_gender(x.split()[0]))

df.loc[df["gender"]=="mostly_female","gender"]="female"
df.loc[df["gender"]=="mostly_male","gender"]="male"

df.loc[df["name"].str.split().str[0].str.lower() == "kim", "gender"] = "male"

## sentida

In [35]:
mysentida = Sentida()

def safe_sentida(x):
    try:
        if not isinstance(x, str) or not x.strip():
            return 0.0
        return mysentida.sentida(x, output="mean", normal=False)
    except Exception:
        return 0.0

df["sentida_score"] = df["content"].apply(safe_sentida)

## lokation

In [36]:
# ── By-ekstraktion via DAWA API ────────────────────────────────────────────────
# Kræver: pip install requests
# ──────────────────────────────────────────────────────────────────────────────


CACHE_FILE = "danske_byer_cache.json"

# Ord der er gyldige stednavne men bruges langt hyppigere som almindelige ord
FALSKE_POSITIVE = {
    # Korte tvetydige bynavne
    'give','lang','hold','giver','skader','måde','vente','ringe','beder',
    'løsning','skade','byen','vente','klippede','nørre','sønder','øster',
    'vester','neder','over','under','lille','store','gamle','nye',
    # Almindelige ord der matcher bynavne
    'marked','have','bank','mark','dal','høj','bakke','eng','holm',
    'strand','havn','vej','gade','plads','torv','slot','kirke','skole',
    # Tvetydige korte navne
    'as','ry','år','vi','ham','her','dem','den','det','fra','mod',
    'ved','til','for','med','men','kan','vil','har','var','kom',
    'tog','sat','gik','lod','fik','bad','bad','bad',
}

# Byer hvor Elgiganten/Power faktisk har butikker — disse prioriteres
# og matcher selv ved kortere navne
KENDTE_KÆDE_BYER = {
    'København','Aarhus','Odense','Aalborg','Esbjerg','Randers','Kolding',
    'Horsens','Vejle','Roskilde','Herning','Silkeborg','Næstved','Fredericia',
    'Viborg','Køge','Holstebro','Slagelse','Hillerød','Sønderborg','Haderslev',
    'Frederikshavn','Hjørring','Holbæk','Svendborg','Helsingør','Frederiksberg',
    'Glostrup','Herlev','Ballerup','Lyngby','Gentofte','Albertslund','Ishøj',
    'Greve','Høje-Taastrup','Høje Taastrup','Brøndby','Hvidovre','Rødovre',
    'Taastrup','Brønshøj','Vanløse','Amager','Kastrup','Tårnby',
    'Skejby','Viby','Brabrand','Åbyhøj','Tranbjerg','Lystrup','Hinnerup',
    'Skive','Thisted','Struer','Lemvig','Ringkøbing','Ikast','Skanderborg',
    'Hadsten','Hammel','Galten','Odder','Hedensted','Frederikssund',
    'Hellerup','Birkerød','Allerød','Farum','Gladsaxe','Søborg',
    'Charlottenlund','Klampenborg','Humlebæk','Espergærde',
    'Middelfart','Nyborg','Svendborg','Faaborg','Assens','Kerteminde',
    'Aabenraa','Tønder','Padborg','Kruså','Gråsten','Nordborg',
    'Nykøbing F','Nykøbing M','Nykøbing Sj','Nakskov','Maribo',
    'Kalundborg','Korsør','Ringsted','Sorø','Lejre',
    'Vordingborg','Faxe','Haslev','Stevns',
    'Grenaa','Ebeltoft','Rønde','Malling','Beder',
    'Rønne','Nexø',
}

def _hent_byer(brug_cache=True):
    if brug_cache and Path(CACHE_FILE).exists():
        with open(CACHE_FILE) as f:
            return json.load(f)
    print("Henter stednavne fra DAWA API...")
    lookup = {}
    for item in requests.get("https://api.dataforsyningen.dk/postnumre?format=json", timeout=15).json():
        n = item['navn'].strip(); lookup[n.lower()] = n
    for item in requests.get("https://api.dataforsyningen.dk/supplerendebynavne2?format=json", timeout=15).json():
        n = item['navn'].strip(); lookup[n.lower()] = n
    side = 1
    while True:
        data = requests.get(
            f"https://api.dataforsyningen.dk/steder?hovedtype=Bebyggelse&format=json&side={side}&per_side=1000",
            timeout=15
        ).json()
        if not data: break
        for item in data:
            n = item.get('primærtnavn', '').strip()
            if n: lookup[n.lower()] = n
        if len(data) < 1000: break
        side += 1
    print(f"  {len(lookup)} stednavne hentet")
    with open(CACHE_FILE, 'w') as f:
        json.dump(lookup, f, ensure_ascii=False)
    return lookup

def _byg_regex(lookup):
    """
    Filtrerer stednavne i to niveauer:
    - Kendte kæde-byer: altid med (selv korte navne)
    - Øvrige: kun hvis længde >= 5 og ikke i falske-positive listen
    """
    filtered = {}
    kendte_lower = {b.lower() for b in KENDTE_KÆDE_BYER}

    for low, canon in lookup.items():
        if low in FALSKE_POSITIVE:
            continue
        if low in kendte_lower:
            filtered[low] = canon  # altid inkluder kendte kæde-byer
        elif len(low) >= 5:
            filtered[low] = canon  # kun lange navne for øvrige

    # Sorter: kendte byer først, derefter længste først
    def sort_key(x):
        low = x[0]
        is_known = low in kendte_lower
        return (0 if is_known else 1, -len(low))

    cities = sorted(filtered.items(), key=sort_key)

    pattern = r'(?<![a-zæøå])(' + '|'.join(re.escape(low) for low, _ in cities) + r')(?![a-zæøå])'
    compiled = re.compile(pattern)
    reverse = {low: canon for low, canon in cities}

    print(f"  {len(cities)} stednavne efter filtrering (ud af {len(lookup)})")
    return compiled, reverse

def find_by(row, pattern, reverse):
    alle = []
    for tekst in [str(row['title']), str(row['content'])]:
        for m in pattern.finditer(tekst.lower()):
            by = reverse.get(m.group(1), m.group(1).title())
            if by not in alle:
                alle.append(by)
    if len(alle) == 0: return "Ukendt by"
    if len(alle) == 1: return alle[0]
    return "Flere byer"

# ── Kør ───────────────────────────────────────────────────────────────────────
_lookup = _hent_byer(brug_cache=True)
_pattern, _reverse = _byg_regex(_lookup)
print(f"Klar\n")

df["by"] = df.apply(lambda row: find_by(row, _pattern, _reverse), axis=1)

print("By-fordeling:")
print(df["by"].value_counts().to_string())


  15733 stednavne efter filtrering (ud af 16196)
Klar

By-fordeling:
by
Ukendt by        3649
Flere byer        329
Kunden             69
Glostrup           44
Vejen              42
Herlev             41
Esbjerg            40
Koster             35
Næstved            34
Kolding            33
Grund              32
Odense             31
Ishøj              28
Herning            25
Hillerød           24
Aalborg            23
Horsens            22
Gentofte           21
Lyngby             20
Roskilde           19
Randers            17
Holstebro          17
Holbæk             17
Tilst              16
Skejby             16
Sønderborg         15
Ringsted           15
Sverige            15
Slagelse           14
Langsom            13
Køge               13
Silkeborg          12
Frederiksberg      12
Viborg             12
Svendborg          11
Hørsholm           11
Fredericia         11
Vejle               9
Taastrup            9
Stilling            8
Virket              7
Viby                7
Hjør

##  måned for review

In [37]:
df["måned"] = pd.to_datetime(df["published"]).dt.month

# **DF to pickle og fjerner kolonner**

In [38]:
df.drop('rating', axis=1, inplace=True)


df.to_pickle("min_færdige_data.pkl")

# **Data exploration**

In [40]:

#df[df["by"] == "Skejby"]["rating"].mean()
print(len(df))



5000
